# Predicción temprana de ansiedad usando datos fisiológicos multimodales
---

## Introducción

El estrés crónico representa uno de los principales factores de riesgo para enfermedades cardiovasculares, depresión y deterioro cognitivo. Su detección temprana en base a datos de la vida diaria es un desafío relevante tanto para la medicina como para el desarrollo de sistemas de salud digital.

En el proyecto utiliza el dataset **SWEET** (*Large-scale wearable data reveal digital phenotypes for daily-life stress detection*), un estudio a gran escala con 1002 sujetos monitoreados durante 5 días consecutivos mediante dispositivos wearables. A partir de una selección de 55 usuarios, se construyó un pipeline de Machine Learning que comprende análisis exploratorio de datos, extracción de características mediante imágenes generadas con TINTOlib, y comparación estadística de dos modelos ML (Random Forest y SVM).

El objetivo es conocer si con estos modelos se pueden crear estrategias para distinguir tres niveles de estrés auto-reportado: **S1** (sin estrés), **S2** (estrés leve) y **S3** (estrés alto), utilizando exclusivamente señales fisiológicas no invasivas (ECG, ACC, TEMP).

---

## 1. Análisis Exploratorio de Datos (EDA)

### ¿Es la data balanceada?

No. El dataset presenta un desbalance significativo entre clases:

| Clase | Muestras | Porcentaje |
|-------|----------|------------|
| S1 — Sin estrés | 443 | 57.5% |
| S2 — Leve | 220 | 28.6% |
| S3 — Alto | 107 | 13.9% |

Este patrón es el mismo al paper original del SWEET Study, donde el 50.4% de los reportes correspondían a ausencia de estrés. El desbalance es normal en la vida real: las personas pasan la mayor parte del tiempo en estados de bajo estrés. Para mitigar su efecto en los modelos, se utilizó el parámetro `class_weight='balanced'` en ambos clasificadores, lo que pondera inversamente el error según la frecuencia de cada clase.
![EDA1](eda_01_distribucion_clases.png)

### ¿Está normalizada la data?

Las señales fisiológicas presentan escalas muy distintas entre sí: la frecuencia cardíaca se mide en bpm (~60–100), la conductancia de piel en µS (~0–20) y la temperatura en °C (~30–36). Se requirió normalización, para que los modelos no asignen mayor peso a las variables con valores más grandes.
![EDA2](eda_02_antes_normalizar.png)

Se aplicó **z-score por usuario**, es decir, cada señal se normaliza restando la media del propio usuario y dividiendo por su desviación estándar. Se lo realiza de esta manera porque cada persona tiene su propia línea base fisiológica — la misma frecuencia cardíaca de 80 bpm puede ser normal para una persona no deportista y elevada para otra. Normalizar por usuario debería disminuir variabilidad y permitir que los modelos aprendan patrones relativos al estado de cada individuo.
![EDA3](eda_03_despues_normalizar.png)

### ¿Hay estacionalidad en las series de tiempo?

Sí. Las señales fisiológicas exhiben ritmos circadianos claros. La frecuencia cardíaca es sistemáticamente más baja durante la noche, mientras que la conductancia de piel y la temperatura presentan patrones inversos. Este comportamiento es consistente con lo reportado en el paper SWEET, que documentó diferencias significativas entre mediciones nocturnas y diurnas (Wilcoxon p < 0.001). La estacionalidad implica que la hora del día es una variable contextual relevante que debería incorporarse como feature adicional en versiones futuras

![EDA 4](eda_04_estacionalidad.png)

### ¿Se necesitan transformaciones no lineales (skewness)?

Algunas features presentaron distribuciones asimétricas con skewness superior a 1, particularmente las relacionadas con la conductancia de piel (GSR) y componentes de variabilidad cardíaca en frecuencia (ECG_LF, ECG_HF). Para estas variables se aplicó la transformación **log1p** (logaritmo de x+1), para comprimir la cola derecha de la distribución acercándola una normal.
![EDA 5](eda_05_skewness.png)


---

## 2. Extracción de Características — TINTOlib y CNN

### TINTOlib

TINTOlib convierte datos tabulares en imágenes 2D, permitiendo aplicar técnicas para datos estructurados. El método utilizado, **TINTO**, proyecta las features numéricas en un espacio 2D mediante PCA para decidir la posición de cada feature en la imagen, y luego asigna un valor de intensidad de píxel proporcional al valor de la feature. El resultado es que cada fila del dataset de features fisiológicas se transforma en una imagen cuadrada (64 x 64) donde la distribución espacial y la intensidad de los píxeles se utilizan para codificar estados fisilogicos.

En este proyecto se utilizaron **12 features fisiológicas** como entrada (mayor correlación):
- ECG: frecuencia cardíaca media, SDNN, RMSSD, LF, HF, LFHF
- Acelerómetro: mean_x, mean_y, mean_z
- Conductancia de piel: GSR_mean
- Temperatura: TEMP_mean
- ACC_mag_mean

Se generaron imágenes de **64×64 píxeles** (un tamaño mayor al default de 20×20 para aportar más información a la CNN), obteniendo **770 imágenes** distribuidas en tres carpetas según la clase de estrés (S1, S2, S3).

### Red Neuronal Convolucional (CNN)

Sobre las imágenes generadas por TINTOlib se entrenó una CNN con el objetivo de extraer **embeddings** — representaciones vectoriales compactas de 32 dimensiones que capturan los patrones aprendidos por la red. La arquitectura utilizada consistió en tres bloques convolucionales con número reducido de filtros (8→16→32), seguidos de una capa de embedding lineal sin función de activación para preservar el rango completo de valores (positivos y negativos). Los pesos se inicializaron con **Xavier uniform** para las capas lineales y **He inicialización** para las capas convolucionales, para evitar embeddings colapsados a cero.

### Limitaciones y trabajo futuro

Los resultados de la CNN mostraron signos de overfitting: el F1-macro en entrenamiento superó 0.9 mientras que en validación se mantuvo por debajo de 0.35. Esto es atribuible principalmente a la **escasez de datos** y que **no se encuentran balanceados** — 770 imágenes es insuficiente para que una CNN aprenda representaciones robustas desde cero. Para mejorar esto:

- **Aumentar el número de usuarios**: incorporar los 101 usuarios disponibles en lugar de 55 para duplicar el dataset
- **Probar otros métodos de TINTOlib**: la librería incluye métodos alternativos como IGTD, SuperTML o DeepInsight, que utilizan estrategias de proyección distintas a PCA y podrían generar mejores imagenes
- **Ampliar el conjunto de features**: incorporar mas features que pueden aportar mas informacion para detectar estrés
- **Transfer learning**: en lugar de entrenar desde cero, usar una red preentrenada en como ResNet18

---

## 3. Optimización de Hiperparámetros

### Métrica de evaluación — F1-macro

Se utilizó el **F1-macro** como métrica principal en lugar del accuracy. El accuracy es engañoso con clases desbalanceadas: un modelo que clasifique todas las muestras como S1 obtendría un accuracy del 57.5% sin haber aprendido nada útil. El F1-score combina precisión y recall en una sola métrica, y la versión **macro** calcula el promedio simple entre clases, dando igual peso a S1, S2 y S3 independientemente de su frecuencia. Esto penaliza explícitamente los modelos que ignoran las clases minoritarias.

### Curvas de optimización de hiperparámetros

**Random Forest** — La gráfica izquierda muestra el F1-macro en función del número de árboles (`n_estimators`) para tres niveles de profundidad máxima. Se observa que `max_depth=5` consistentemente supera a `max_depth=None` y `max_depth=10`. Esto es contraintuitivo: normalmente más árboles sin restricción de profundidad mejora el rendimiento, pero en este caso la restricción de profundidad actúa como **regularización**, evitando que cada árbol memorice el conjunto de entrenamiento. El mejor rendimiento se alcanza con pocos árboles (25–50) y `max_depth=5`, lo que sugiere que el modelo se beneficia de mayor simplicidad dada la cantidad limitada de datos.

**SVM** — La gráfica derecha muestra el efecto del parámetro de regularización `C` para kernels RBF y lineal. El kernel lineal mantiene un rendimiento estable (~0.31) a lo largo de todos los valores de C, mientras que el kernel RBF mejora progresivamente con C más alto, alcanzando ~0.34 en C=10. Esto indica que la frontera de decisión óptima para este problema es no lineal y que el kernel RBF captura mejor la estructura de los embeddings, aunque a riesgo de sobreajuste con valores de C muy altos.

![Hiperparametros](hiperparametros.png)

### Curvas de aprendizaje

**Random Forest** — La curva muestra un patrón clásico de **alta varianza (overfitting)**. El F1-macro en entrenamiento comienza en 1.0 y decrece lentamente a medida que se añaden más muestras, mientras que la curva de validación sube muy lentamente desde 0.25 hasta ~0.37. La brecha entre ambas curvas es grande y no converge, lo que indica que el modelo memoriza los datos de entrenamiento pero no generaliza bien. La solución indicada para este patrón es **más datos de entrenamiento** — si las curvas están todavía divergiendo al usar el 80% del dataset, agregar más muestras debería mejorar la validación.

**SVM** — Presenta un comportamiento diferente: la curva de entrenamiento comienza en ~0.68 y se mantiene relativamente estable, mientras que la validación sube levemente desde 0.30 hasta ~0.36. La brecha es menor que en RF pero persiste, indicando también **overfitting moderado**. El hecho de que la curva de train de SVM no llegue a 1.0 sugiere que SVM tiene mayor regularización implícita que RF, lo cual es consistente con su naturaleza de maximización del margen.

En ambos modelos, las curvas de validación muestran una tendencia ascendente que no ha convergido al usar todas las muestras disponibles. Esto es evidencia directa de que **el modelo mejoraría con más datos** — no es un problema de la arquitectura del modelo sino del tamaño del dataset.

![aprendizaje](aprendizaje.png)

### Comparación estadística — Test de Wilcoxon

Para comparar los dos modelos se utilizó el **test de Wilcoxon de rangos con signo**, que es el test no paramétrico recomendado para comparar clasificadores en Machine Learning. Se prefiere sobre el t-test porque no asume normalidad en la distribución de los scores, lo cual es importante cuando el número de evaluaciones es pequeño (10 folds en este caso). El test compara si la diferencia entre los scores de RF y SVM, fold por fold, es sistemáticamente mayor en un sentido o en el otro.

Los resultados obtenidos fueron:
- Random Forest: F1-macro = 0.346 ± 0.060
- SVM: F1-macro = 0.342 ± 0.049
- Estadístico W = 27.0, **p-value = 1.0**

El p-value de 1.0 indica que no existe ninguna diferencia sistemática entre los scores de ambos modelos — en la mayoría de los folds obtienen exactamente el mismo rendimiento. No hay evidencia estadística para preferir uno sobre el otro. Ante este resultado, se recomienda **Random Forest** por ser más interpretable: permite extraer la importancia de cada embedding y ofrece mayor robustez ante datos ruidosos.

---

## 4. Visualización t-SNE sobre Embeddings CNN

La visualización t-SNE reduce los 32 embeddings a 2 dimensiones para evaluar visualmente si la CNN aprendió representaciones que separan los niveles de estrés.

Al observar la figura, se aprecia que los tres niveles de estrés **no forman grupos claramente separados** — los puntos de S1 (verde), S2 (naranja) y S3 (rojo) se encuentran mezclados a lo largo de todo el espacio 2D. Sin embargo, se pueden observar algunas estructuras locales de interés: en la región superior izquierda existe una concentración de puntos S2 y S3 que sugiere que la CNN capturó algo de señal diferenciadora para estados de estrés elevado, aunque insuficiente para una separación limpia.

Esta visualización es consistente con los resultados de clasificación: un F1-macro de ~0.33–0.35 corresponde a un espacio de embeddings donde las clases se superponen significativamente. Comparado con un t-SNE ideal donde cada clase formaría un cluster compacto y separado, los embeddings actuales capturan estructura parcial pero no discriminativa.

La mezcla de clases en el espacio de embeddings puede atribuirse a dos factores: primero, la naturaleza inherentemente difusa del estrés auto-reportado (la misma respuesta fisiológica puede corresponder a distintos niveles subjetivos de estrés); segundo, la limitada capacidad de la CNN para aprender representaciones robustas con 770 imágenes.

---

## 5. Conclusiones y Trabajo Futuro

### Conclusiones

El pipeline desarrollado — preprocesamiento de señales fisiológicas, conversión a imágenes con TINTOlib, extracción de embeddings con CNN y clasificación con Random Forest y SVM — es puede ser viable pero por el momento tiene rendimiento limitado por el tamaño del dataset.

Los modelos obtuvieron un F1-macro de ~0.34, superando ligeramente la clasificación aleatoria (~0.33 para 3 clases desbalanceadas) pero sin alcanzar el F1=0.43 reportado en el paper SWEET con Random Forest sobre features directas. Esto sugiere que el pipeline de TINTOlib→CNN introduce ruido adicional cuando el dataset es pequeño, pero deberia intentarse con mas datos.

Las curvas de aprendizaje evidencian **alta varianza (overfitting)** en ambos modelos, con una brecha persistente entre train y validación que no converge. Este patrón es la evidencia más clara de que el modelo necesita más datos antes de explorar otros modelos. El test de Wilcoxon (p=1.0) indica que RF y SVM son estadísticamente equivalentes sobre estos embeddings, por lo que se recomienda Random Forest por su mayor interpretabilidad.

### Propuestas de mejora

**1. Aumentar el volumen de datos**
Las curvas de aprendizaje muestran que la validación sigue mejorando al agregar datos — aún no ha convergido. Incorporar los 101 usuarios disponibles (en lugar de 55) y explorar ventanas de tiempo más amplias o múltiples ventanas por reporte podría duplicar o triplicar el dataset.

**2. Enriquecer el conjunto de features**
El paper SWEET identificó SC_phasic y SC_area como las features más discriminativas para detectar estrés. Estas requieren un procesamiento mas extenso como descomponer la señal GSR componentes tónica y fásica.

**3. Explorar otros métodos de TINTOlib**
TINTOlib incluye múltiples métodos de conversión: IGTD, SuperTML, DeepInsight y BarGraph, entre otros. Cada uno utiliza una estrategia distinta para proyectar las features en el espacio 2D, lo que puede resultar en imágenes con diferente información util para el objetivo

**4. Transfer learning sobre imágenes TINTOlib**
En lugar de entrenar una CNN desde cero, aplicar transfer learning con modelos preentrenados

**5. Explorar modelos fundacionales para series temporales — TimesFM**
Planea explorar **TimesFM**, un modelo fundacional pre-entrenado por Google para series temporales. recibe directamente las señales fisiológicas como series temporales y puede incorporar variables exógenas como la hora del día, la actividad física y el contexto del usuario. Los modelos fundacionales tienen la ventaja de haber sido pre-entrenados en grandes volúmenes de datos temporales, lo que les permite generalizar con menor cantidad de datos. Dado que el principal limitante de este proyecto fue el tamaño del dataset, TimesFM representa una alternativa para superar el rendimiento de la pipeline actual.
